# 🎬 Spec-FastGS Video Demo Generator: Scene 'counter'

Notebook này tự động hóa toàn bộ quy trình trích xuất dữ liệu làm **Video Clip Demo** cho đề tài Spec-FastGS trên Kaggle:
1. **Khâu 0 (Cấu hình & Chọn Góc Chụp)**: Tự động chọn cố định 10 góc ảnh (10 fixed camera viewpoints) đại diện cho scene `counter`.
2. **Khâu 1 (Mây điểm ban đầu - Initial Point Cloud)**: Render và lưu ảnh mây điểm thưa ban đầu (Iteration 0) từ đúng 10 góc chụp cố định.
3. **Khâu 2 (Reflection Prior Maps)**: Trích xuất 10 bản đồ Reflection Prior (phương pháp Tan-Ikeuchi) tương ứng với 10 góc chụp.
4. **Khâu 3 (Huấn luyện & Capture Snapshots mỗi 1k iter)**: Huấn luyện Spec-fastgs 30.000 iterations, sau mỗi **1.000 iterations** tự động render và lưu 10 khung ảnh từ 10 góc chụp cố định.
5. **Khâu 4 (Tự động Upload Hugging Face)**: Đẩy toàn bộ ảnh và kết quả lên HuggingFace Dataset: **`DiBiay/video-demo-thesis`**.

In [ ]:
# ── Cấu Hình Biến Môi Trường & Token ────────────────────────────────────────────
import os
import sys

PROJECT_DIR = "/kaggle/working/thesis-all"
DEMO_OUTPUT_DIR = "/kaggle/working/demo_video_counter"
HF_REPO_ID = "DiBiay/video-demo-thesis"

# Token Hugging Face (Vui lòng điền HF Token của bạn nếu chưa thiết lập biến môi trường)
HF_TOKEN = os.environ.get("HF_TOKEN", "YOUR_HF_TOKEN") #@param {type:"string"}
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

os.makedirs(DEMO_OUTPUT_DIR, exist_ok=True)
print(f"✅ Cấu hình hoàn tất!")
print(f"Thư mục kết quả Demo: {DEMO_OUTPUT_DIR}")

In [ ]:
# ── Clone Repo & Thiết Lập Dataset 'counter' ───────────────────────────
import os
import shutil

REPO_URL = "https://github.com/0Nguyen0Cong0Tuan0/thesis-all.git"
KAGGLE_INPUT_DATASET = "/kaggle/input/mipnerf360-dataset"

# Đảm bảo không bị lỗi đã tồn tại folder dở dang khi git clone
specfastgs_check = os.path.join(PROJECT_DIR, "spec-fastgs")
if not os.path.exists(specfastgs_check):
    print(f"📦 Đang clone repository từ {REPO_URL} ...")
    if os.path.exists(PROJECT_DIR) and not os.path.exists(os.path.join(PROJECT_DIR, ".git")):
        shutil.rmtree(PROJECT_DIR, ignore_errors=True)
    !git clone --recursive {REPO_URL} "{PROJECT_DIR}"
else:
    print(f"✅ Repo đã tồn tại tại {PROJECT_DIR}")

# Tạo cấu hình dataset cho scene counter
target_scene_dir = os.path.join(PROJECT_DIR, "spec-fastgs", "datasets", "mipnerf360", "counter")
source_scene_dir = os.path.join(KAGGLE_INPUT_DATASET, "counter")

os.makedirs(target_scene_dir, exist_ok=True)

if os.path.exists(source_scene_dir):
    for item in os.listdir(source_scene_dir):
        src_item = os.path.join(source_scene_dir, item)
        dst_item = os.path.join(target_scene_dir, item)
        # 1. Thư mục ảnh (images_8, images_4, images): Tạo Symlink để đọc nhanh
        if os.path.isdir(src_item) and item in ["images_8", "images_4", "images"]:
            if os.path.lexists(dst_item):
                os.remove(dst_item) if not os.path.isdir(dst_item) else shutil.rmtree(dst_item)
            os.symlink(src_item, dst_item)
            print(f"  -> Created symlink for images: {dst_item} -> {src_item}")
        # 2. Thư mục 'sparse': COPY PHYSICALLY để ghi file points3D.ply vào /kaggle/working mà không bị lỗi Read-only
        elif os.path.isdir(src_item) and item == "sparse":
            if os.path.lexists(dst_item):
                os.remove(dst_item) if not os.path.isdir(dst_item) else shutil.rmtree(dst_item)
            shutil.copytree(src_item, dst_item)
            print(f"  -> Physically copied 'sparse' folder to writable location: {dst_item}")
        elif os.path.isfile(src_item):
            if not os.path.exists(dst_item):
                shutil.copy2(src_item, dst_item)
    print("✅ Thiết lập Dataset cho scene 'counter' thành công!")
else:
    print(f"❌ Không tìm thấy scene 'counter' trong {KAGGLE_INPUT_DATASET}")

In [ ]:
# ── Cài Đặt Official COLMAP Binary (từ colmap/colmap) & Thư Viện Phụ Thuộc ────────
!apt-get update -qq && apt-get install -y -qq colmap
!colmap --help > /dev/null && echo "✅ Official COLMAP binary successfully installed!"
!pip install -q plyfile tqdm websockets openpyxl pandas ninja huggingface_hub

import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")

# Tự động thiết lập CUDA toolkit
cuda_dirs = ["/usr/local/cuda", "/usr/local/cuda-12.2", "/usr/local/cuda-12.1", "/usr/local/cuda-12", "/usr/local/cuda-11.8"]
detected_cuda = None
for d in cuda_dirs:
    if os.path.exists(os.path.join(d, "bin", "nvcc")):
        detected_cuda = d
        break
if detected_cuda:
    os.environ["CUDA_HOME"] = detected_cuda
    os.environ["PATH"] = f"{detected_cuda}/bin:" + os.environ.get("PATH", "")
    os.environ["LD_LIBRARY_PATH"] = f"{detected_cuda}/lib64:" + os.environ.get("LD_LIBRARY_PATH", "")
    print(f"Configured CUDA: {detected_cuda}")

submodules_dir = os.path.join(PROJECT_DIR, "FastGS_backup_v2", "submodules")
if os.path.exists(submodules_dir):
    !cd "{submodules_dir}/diff-gaussian-rasterization_fastgs" && pip install -q .
    !cd "{submodules_dir}/simple-knn" && pip install -q .
    !cd "{submodules_dir}/fused-ssim" && pip install -q .
    print("🎉 Biên dịch và nạp CUDA Submodules thành công!")

In [ ]:
# ── KHÂU 0 & KHÂU 1: Chạy COLMAP Tạo Mây Điểm Ban Đầu & Render 10 Góc Chụp (Iter 0) ───
import os
import sys
import json
import torch
import subprocess
import shutil
from PIL import Image

# Ép buộc Qt chạy ở chế độ offscreen cho máy chủ headless (Kaggle/Colab)
os.environ["QT_QPA_PLATFORM"] = "offscreen"

specfastgs_path = os.path.abspath(os.path.join(PROJECT_DIR, "spec-fastgs"))
if not os.path.exists(specfastgs_path):
    raise RuntimeError(f"❌ Không tìm thấy thư mục spec-fastgs tại {specfastgs_path}. Vui lòng chạy lại Cell 3 (git clone).")

if specfastgs_path not in sys.path:
    sys.path.insert(0, specfastgs_path)
os.chdir(specfastgs_path)

source_path = os.path.join(specfastgs_path, "datasets", "mipnerf360", "counter")
step1_dir = os.path.join(DEMO_OUTPUT_DIR, "step1_initial_points")
demo_model_path = os.path.join(PROJECT_DIR, "spec-fastgs", "output", "demo_counter")
os.makedirs(step1_dir, exist_ok=True)
os.makedirs(demo_model_path, exist_ok=True)

# 1. Tái tạo Mây Điểm Ban Đầu (Sparse Point Cloud) bằng COLMAP từ tập ảnh đầu vào
colmap_db = os.path.join(source_path, "database.db")
images_dir = os.path.join(source_path, "images_8")
if not os.path.exists(images_dir):
    images_dir = os.path.join(source_path, "images")

print(f"🌐 Đang chạy COLMAP tái tạo mây điểm từ tập ảnh đầu vào: {images_dir} ...")
colmap_env = os.environ.copy()
colmap_env["QT_QPA_PLATFORM"] = "offscreen"

try:
    if os.path.exists(colmap_db):
        os.remove(colmap_db)
    print("  [1/3] Trích xuất SIFT features (CPU mode for headless server)...")
    subprocess.run([
        "colmap", "feature_extractor",
        "--database_path", colmap_db,
        "--image_path", images_dir,
        "--ImageReader.single_camera", "1",
        "--SiftExtraction.use_gpu", "0"
    ], env=colmap_env, check=True)
    print("  [2/3] Ghép nối features (Exhaustive Matcher)..." )
    subprocess.run([
        "colmap", "exhaustive_matcher",
        "--database_path", colmap_db,
        "--SiftMatching.use_gpu", "0"
    ], env=colmap_env, check=True)
    print("  [3/3] Dựng mây điểm thưa 3D (COLMAP Mapper)..." )
    sparse_output = os.path.join(source_path, "sparse")
    os.makedirs(os.path.join(sparse_output, "0"), exist_ok=True)
    subprocess.run([
        "colmap", "mapper",
        "--database_path", colmap_db,
        "--image_path", images_dir,
        "--output_path", sparse_output,
        "--Mapper.ba_global_function_tolerance", "0.000001"
    ], env=colmap_env, check=True)
    print("✅ COLMAP tái tạo Mây Điểm Ban Đầu (Sparse Point Cloud) thành công!")
except Exception as e:
    print(f"⚠️ Lưu ý khi chạy COLMAP tái tạo mây điểm: {e}")
    print("👉 Sử dụng mây điểm sẵn có trong thư mục sparse writable để tiếp tục render.")

# 2. Khởi tạo cấu hình GaussianModel & Scene
from argparse import ArgumentParser
from scene import Scene, GaussianModel
from arguments import ModelParams, PipelineParams
from gaussian_renderer import render_fastgs

arg_parser = ArgumentParser(description="Demo video initial points renderer")
lp = ModelParams(arg_parser, sentinel=True)
pp = PipelineParams(arg_parser)
parsed_args = arg_parser.parse_args(["-s", source_path, "-m", demo_model_path, "-i", "images_8", "--sh_degree", "3", "--asg_degree", "24", "--resolution", "-1", "--eval"])
dataset = lp.extract(parsed_args)
pipe = pp.extract(parsed_args)

gaussians = GaussianModel(dataset.sh_degree, dataset.asg_degree)
scene = Scene(dataset, gaussians, load_iteration=None, shuffle=False)

test_cameras = scene.getTestCameras()
num_cams = len(test_cameras)
print(f"Tổng số camera test khả dụng trong scene 'counter': {num_cams}")

# Chọn 10 góc ảnh phân bố đều
selected_indices = [int(i * (num_cams - 1) / 9) for i in range(10)]
selected_cameras = [test_cameras[idx] for idx in selected_indices]
selected_cam_names = [cam.image_name for cam in selected_cameras]

# Lưu thông tin 10 góc camera chọn cố định
fixed_cams_path = os.path.join(DEMO_OUTPUT_DIR, "fixed_cameras.json")
with open(fixed_cams_path, "w", encoding="utf-8") as f:
    json.dump({"selected_indices": selected_indices, "camera_names": selected_cam_names}, f, indent=4)

print(f"✅ Đã chọn cố định 10 góc chụp camera: {selected_cam_names}")

# Render mây điểm ban đầu (Iteration 0)
bg_color = torch.tensor([0, 0, 0], dtype=torch.float32, device="cuda")

print("🎨 Đang render và lưu 10 ảnh mây điểm thưa ban đầu (Khâu 1)..." )
with torch.no_grad():
    for i, cam in enumerate(selected_cameras):
        render_pkg = render_fastgs(cam, gaussians, pipe, bg_color, 1.0)
        img_tensor = render_pkg["render"].clamp(0.0, 1.0)
        img_np = (img_tensor.detach().permute(1, 2, 0).cpu().numpy() * 255).astype("uint8")
        out_file = os.path.join(step1_dir, f"view_{i+1:02d}_{cam.image_name}.png")
        Image.fromarray(img_np).save(out_file)
        print(f"  -> Đã lưu Khâu 1 ảnh {i+1}/10: {out_file}")

print("✅ Khâu 1 hoàn tất!")

In [ ]:
# ── KHÂU 2: Trích Xuất & Lưu 10 Bản Đồ Reflection Prior (Tan-Ikeuchi) ────────────────
import os
import sys
import json
import shutil
import subprocess

source_path = os.path.join(PROJECT_DIR, "spec-fastgs", "datasets", "mipnerf360", "counter")
step2_dir = os.path.join(DEMO_OUTPUT_DIR, "step2_reflection_priors")
os.makedirs(step2_dir, exist_ok=True)

# 1. Chạy script trích xuất reflection prior theo phương pháp Tan-Ikeuchi
print("🔍 Đang chạy trích xuất Reflection Prior bản đồ bóng (Tan-Ikeuchi)..." )
cmd = [
    sys.executable, "extract_reflection_prior.py",
    "-s", source_path,
    "-i", "images_8",
    "--ref_prior_method", "tan"
]
subprocess.run(cmd, cwd=os.path.join(PROJECT_DIR, "spec-fastgs"), check=True)

# 2. Đọc lại danh sách 10 camera cố định từ Khâu 0
fixed_cams_path = os.path.join(DEMO_OUTPUT_DIR, "fixed_cameras.json")
with open(fixed_cams_path, "r", encoding="utf-8") as f:
    cam_data = json.load(f)
selected_cam_names = cam_data["camera_names"]

# Thư mục chứa prior được trích xuất ra
prior_dir = os.path.join(source_path, "reflection_prior")

print("📸 Đang trích xuất và sao chép 10 bản đồ Reflection Prior tương ứng...")
for i, cam_name in enumerate(selected_cam_names):
    # Tìm file prior tương ứng
    prior_file = os.path.join(prior_dir, f"{cam_name}.png")
    if not os.path.exists(prior_file):
        # Nếu định dạng file chứa đuôi jpeg/jpg
        prior_file = os.path.join(prior_dir, f"{cam_name}.jpg")
        
    if os.path.exists(prior_file):
        out_file = os.path.join(step2_dir, f"view_{i+1:02d}_{cam_name}_prior.png")
        shutil.copy2(prior_file, out_file)
        print(f"  -> Đã lưu Khâu 2 Prior {i+1}/10: {out_file}")
    else:
        print(f"  ⚠️ Cảnh báo: Không tìm thấy prior cho camera {cam_name} tại {prior_file}")

print("✅ Khâu 2 hoàn tất!")

In [ ]:
# ── KHÂU 3: Huấn Luyện 30k Iterations & Snapshots 10 Góc Chụp Mỗi 5.000 Iterations ────
import os
import sys
import subprocess

source_path = os.path.join(PROJECT_DIR, "spec-fastgs", "datasets", "mipnerf360", "counter")
model_path = os.path.join(PROJECT_DIR, "spec-fastgs", "output", "demo_counter")
step3_dir = os.path.join(DEMO_OUTPUT_DIR, "step3_training_snapshots")
os.makedirs(step3_dir, exist_ok=True)

# Thiết lập biến môi trường DEMO_SNAPSHOT_DIR & DEMO_SNAPSHOT_INTERVAL (5000 iter)
env = os.environ.copy()
env["DEMO_SNAPSHOT_DIR"] = step3_dir
env["DEMO_SNAPSHOT_INTERVAL"] = "5000"

print(f"🏋️ Khởi động huấn luyện Spec-fastgs 30.000 iterations trên scene 'counter'...")
print(f"Cấu hình: Render & lưu 10 góc ảnh snapshot sau mỗi 5.000 iterations (5k, 10k, 15k, 20k, 25k, 30k)")

cmd = [
    sys.executable, "train.py",
    "-s", source_path,
    "-m", model_path,
    "-i", "images_8",
    "--eval",
    "--iterations", "30000",
    "--densification_interval", "100",
    "--optimizer_type", "default",
    "--asg_degree", "64",
    "--is_real", "--is_indoor",
    "--sh_degree", "3",
    "--highfeature_lr", "0.02",
    "--grad_abs_thresh", "0.0004",
    "--specular_start_iter", "3000",
    "--ref_prior_method", "tan",
    "--sh_spec_grad_scale", "0.75",
    "--sh_spec_mask_start", "8000",
    "--sh_spec_mask_threshold", "0.75",
    "--sh_spec_min_metric_count", "2",
    "--use_ref_score", "--use_adaptive_prior", "--use_sh_spec_mask"
]

subprocess.run(cmd, cwd=os.path.join(PROJECT_DIR, "spec-fastgs"), env=env, check=True)
print("✅ Khâu 3 hoàn tất!")

In [ ]:
# ── KHÂU 4: Upload Tự Động Toàn Bộ Kết Quả Lên HuggingFace Dataset ─────────────────
import os
import time
from huggingface_hub import HfApi

HF_REPO_ID = "DiBiay/video-demo-thesis"
DEMO_OUTPUT_DIR = "/kaggle/working/demo_video_counter"
HF_TOKEN = os.environ.get("HF_TOKEN", "")

if not HF_TOKEN or HF_TOKEN == "YOUR_HF_TOKEN":
    print("⚠️ Cảnh báo: Chưa cấu hình HF_TOKEN hợp lệ. Vui lòng kiểm tra lại Cell 2.")
else:
    print(f"☁️ Đang tải toàn bộ thư mục demo ({DEMO_OUTPUT_DIR}) lên HuggingFace Repo: {HF_REPO_ID} ...")
    api = HfApi(token=HF_TOKEN)
    api.create_repo(repo_id=HF_REPO_ID, repo_type="dataset", exist_ok=True)
    
    for attempt in range(3):
        try:
            api.upload_folder(
                folder_path=DEMO_OUTPUT_DIR,
                path_in_repo="counter_demo_video",
                repo_id=HF_REPO_ID,
                repo_type="dataset"
            )
            print(f"🎉 CHÚC MỪNG! Đã upload thành công toàn bộ dữ liệu Video Demo lên Hugging Face!")
            print(f"👉 Link Dataset: https://huggingface.co/datasets/{HF_REPO_ID}")
            break
        except Exception as e:
            print(f"⚠️ Thử lại lần {attempt+1}: Lỗi khi upload: {e}")
            time.sleep(10)